# Librerias

In [96]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib
import json
from tqdm import tqdm
import time


from surprise import Dataset as SurpriseDataset
from surprise import Reader
from surprise import SVDpp
from surprise import accuracy

from sklearn.preprocessing import MinMaxScaler

from typing import Optional, Union

# Cargar los datos

In [97]:
# ── Paths
ROOT       = Path('..')
DATA_DIR   = ROOT / 'data'
RAW_DIR   = DATA_DIR / 'raw' # yelp JSONs live here
PROCESSED_DIR = DATA_DIR / 'processed' # processed data will be saved here
MODEL_DIR = ROOT / 'models' # trained model will be saved here

# create directories if they don't exist
for directory in [DATA_DIR, RAW_DIR, PROCESSED_DIR, MODEL_DIR]:
    if not directory.exists():
        directory.mkdir()

Dado que los archivos son gigantes, se decidió cargar por chunks (1 millón de lineas cada uno, en total son alrededor de 7 chunks para el archivo más grande), y luego guardar en formato parquet para luego re-cargarlo completo en parquet, parece redundante pero ahorra mucha ram y tiempo en los archivos pesados

In [98]:
raw_json_paths = list(RAW_DIR.glob('*.json'))
create_parquets = any(not (RAW_DIR / f"{json_path.stem}.parquet").exists() for json_path in raw_json_paths)
if create_parquets:
    # --- IGNORE ---
    for json_path in raw_json_paths:
        with open(json_path, 'r') as f:
            
            chunksize = 1e6
            yelp_data = []
            yelp_chunks = pd.read_json(json_path, lines=True, chunksize=chunksize)
            for chunk in tqdm(yelp_chunks):
                yelp_data.append(chunk)

            # guardar en parquet
            pd.concat(yelp_data).to_parquet(Path(json_path.parent,f"{json_path.stem}.parquet"), index=False)
            #yelp_data:pd.DataFrame = pd.read_parquet(Path(json_path.parent,f"{json_path.stem}.parquet"))
else:
    print("Parquet files already exist. Skipping JSON to Parquet conversion.")
    print("Loading businesses")
    df_businesses = pd.read_parquet(RAW_DIR / 'yelp_academic_dataset_business.parquet', engine='fastparquet')
    print(f"hay {df_businesses.shape[0]} filas en businesses")
    print(df_businesses.columns, "\n")

    print("Loading reviews")
    df_reviews = pd.read_parquet(RAW_DIR / 'yelp_academic_dataset_review.parquet', engine='fastparquet')
    print(f"hay {df_reviews.shape[0]} filas en reviews")
    print(df_reviews.columns, "\n")
    
    print("Loading users")
    df_users = pd.read_parquet(RAW_DIR / 'yelp_academic_dataset_user.parquet', engine='fastparquet')
    print(f"hay {df_users.shape[0]} filas en users")
    print(df_users.columns, "\n")

Parquet files already exist. Skipping JSON to Parquet conversion.
Loading businesses
hay 150346 filas en businesses
Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'categories', 'attributes.ByAppointmentOnly',
       'attributes.BusinessAcceptsCreditCards', 'attributes.BikeParking',
       'attributes.RestaurantsPriceRange2', 'attributes.CoatCheck',
       'attributes.RestaurantsTakeOut', 'attributes.RestaurantsDelivery',
       'attributes.Caters', 'attributes.WiFi', 'attributes.BusinessParking',
       'attributes.WheelchairAccessible', 'attributes.HappyHour',
       'attributes.OutdoorSeating', 'attributes.HasTV',
       'attributes.RestaurantsReservations', 'attributes.DogsAllowed',
       'attributes.Alcohol', 'attributes.GoodForKids',
       'attributes.RestaurantsAttire', 'attributes.Ambience',
       'attributes.RestaurantsTableService',
       'attributes.RestaurantsGoodForGroup

In [99]:
hours_columns = list(df_businesses.filter(regex='hours.').columns)

# Diccionario con los horarios de cada negocio
df_businesses["hours"] = df_businesses.apply(lambda row: {day.split(".")[1]: row[day] for day in hours_columns if pd.notna(row[day])}, axis=1)
df_businesses[["name", "hours"] + hours_columns ].head()

,name,hours,hours.Monday,hours.Tuesday,hours.Wednesday,hours.Thursday,hours.Friday,hours.Saturday,hours.Sunday
0,"Abby Rappoport, LAC, CMQ",{},None,None,None,None,None,None,None
1,The UPS Store,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ...",0:0-0:0,8:0-18:30,8:0-18:30,8:0-18:30,8:0-18:30,8:0-14:0,None
2,Target,"{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ...",8:0-22:0,8:0-22:0,8:0-22:0,8:0-22:0,8:0-23:0,8:0-23:0,8:0-22:0
3,St Honore Pastries,"{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ...",7:0-20:0,7:0-20:0,7:0-20:0,7:0-20:0,7:0-21:0,7:0-21:0,7:0-21:0
4,Perkiomen Valley Brewery,"{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2...",None,None,14:0-22:0,16:0-22:0,12:0-22:0,12:0-22:0,12:0-18:0


In [100]:
attributes_columns = df_businesses.filter(regex='attributes.').columns
print(attributes_columns)
df_businesses[["name"] + list(attributes_columns)].head()

Index(['attributes.ByAppointmentOnly', 'attributes.BusinessAcceptsCreditCards',
       'attributes.BikeParking', 'attributes.RestaurantsPriceRange2',
       'attributes.CoatCheck', 'attributes.RestaurantsTakeOut',
       'attributes.RestaurantsDelivery', 'attributes.Caters',
       'attributes.WiFi', 'attributes.BusinessParking',
       'attributes.WheelchairAccessible', 'attributes.HappyHour',
       'attributes.OutdoorSeating', 'attributes.HasTV',
       'attributes.RestaurantsReservations', 'attributes.DogsAllowed',
       'attributes.Alcohol', 'attributes.GoodForKids',
       'attributes.RestaurantsAttire', 'attributes.Ambience',
       'attributes.RestaurantsTableService',
       'attributes.RestaurantsGoodForGroups', 'attributes.DriveThru',
       'attributes.NoiseLevel', 'attributes.GoodForMeal',
       'attributes.BusinessAcceptsBitcoin', 'attributes.Smoking',
       'attributes.Music', 'attributes.GoodForDancing',
       'attributes.AcceptsInsurance', 'attributes.BestNights',


,name,attributes.ByAppointmentOnly,attributes.BusinessAcceptsCreditCards,attributes.BikeParking,attributes.RestaurantsPriceRange2,attributes.CoatCheck,attributes.RestaurantsTakeOut,attributes.RestaurantsDelivery,attributes.Caters,attributes.WiFi,...,attributes.AcceptsInsurance,attributes.BestNights,attributes.BYOB,attributes.Corkage,attributes.BYOBCorkage,attributes.HairSpecializesIn,attributes.Open24Hours,attributes.RestaurantsCounterService,attributes.AgesAllowed,attributes.DietaryRestrictions
0,"Abby Rappoport, LAC, CMQ",True,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,The UPS Store,None,True,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,Target,False,True,True,2,False,False,False,False,u'no',...,None,None,None,None,None,None,None,None,None,None
3,St Honore Pastries,False,False,True,1,None,True,False,True,u'free',...,None,None,None,None,None,None,None,None,None,None
4,Perkiomen Valley Brewery,None,True,True,None,None,True,None,False,None,...,None,None,None,None,None,None,None,None,None,None


In [101]:
df_reviews["stars"].value_counts()

stars
5    3231627
4    1452918
1    1069561
3     691934
2     544240
Name: count, dtype: int64

In [ ]:
# modelo de ejemplo con SVD++
model = joblib.load(MODEL_DIR / 'svdpp_model.joblib')

# Contexto

In [103]:
# Convertir categorías a lista limpia
df_businesses["category_list"] = (
    df_businesses["categories"]
    .fillna("")
    # Convertir la cadena de categorías en una lista, manejando casos nulos o vacíos
    .apply(lambda x: x.split(",") if isinstance(x, str) else [])
    # Normalizar categorías: eliminar espacios, convertir a minúsculas y reemplazar espacios por guiones bajos
    .apply(lambda categories: [category.strip().lower().replace(' ', '_') for category in categories])
)

# asignar sin_categoría a negocios sin categorías
df_businesses["category_list"] = df_businesses["category_list"].apply(lambda cats: cats if cats != [''] else ['sin_categoría'])

# eliminar negocios sin categorías
filter_sin_categoria = df_businesses["category_list"].apply(lambda cats: cats == ['sin_categoría'])
print(f"Negocios sin categoría: {filter_sin_categoria.sum()} ({filter_sin_categoria.mean() * 100:.2f}%)")
df_businesses = df_businesses[~filter_sin_categoria].copy()

unique_categories = set(df_businesses["category_list"].explode().unique())
print(f"Unique categories: {len(unique_categories)}")

Negocios sin categoría: 103 (0.07%)
Unique categories: 1311


In [104]:
df_businesses["city"] = df_businesses["city"].fillna("unknown").str.strip().str.lower()

### Popularidad

In [105]:
# Usar log1p para manejar la distribución sesgada de review_count y luego escalar a [0, 1]
df_businesses['pop_score'] = MinMaxScaler().fit_transform(
    np.log1p(df_businesses[['review_count']])
)

df_businesses[['business_id', 'review_count', 'pop_score']].head()

,business_id,review_count,pop_score
0,Pns2l4eNsfO8kk83dixA6A,7,0.040291
1,mpf3x-BjTdTEA3yCZrAYPw,15,0.137370
2,tUFrWirKiKi_TAnsVWINQQ,22,0.188197
3,MTSW4McQd7CbVtyjqoe9mw,80,0.364519
4,mWMc6_wTdE0EUBKIGXDVfA,13,0.118668


In [106]:
def average_rating_weighted_by_user_activity(df_ratings, df_items):
    """Calcula el promedio bayesiano de las calificaciones de las items, 
    ponderado por la actividad de los usuarios."""
    items_stats = df_ratings.groupby('business_id')['stars'].agg({'count','mean','var'}).reset_index()

    # 1. Calcular C (el promedio global de todas las calificaciones)
    C = items_stats['mean'].mean()

    # 2. Calcular m (el mínimo de votos requeridos). ``
    # Aquí usamos el percentil 90: el item debe tener más votos que el 90% de la base de datos
    m = items_stats['count'].quantile(0.90)

    print(f"Promedio global ratings (C): {C:.2f}")
    print(f"Cantidad de ratings mínimos (m) - Percentil 90: {m}")


    # 3. Definir la función del Promedio Bayesiano
    def weighted_rating(x, m=m, C=C):
        v = x['count']
        R = x['mean']
        # Fórmula
        return (v/(v+m) * R) + (m/(m+v) * C)
    
    items_stats['score_bayesiano'] = items_stats.apply(weighted_rating, axis=1)
    return items_stats[['score_bayesiano', 'business_id']].merge(df_items, on='business_id', how='left')

items_with_bayesian_score = average_rating_weighted_by_user_activity(df_reviews, df_businesses)
items_with_bayesian_score.sort_values('score_bayesiano', ascending=False)[['business_id', 'score_bayesiano', 'name', 'review_count']].head(10)

Promedio global ratings (C): 3.59
Cantidad de ratings mínimos (m) - Percentil 90: 101.0


,business_id,score_bayesiano,name,review_count
150248,zxIF-bnaJ-eKIsznB7yu7A,4.712078,Free Tours By Foot,769.0
34320,DVBJRvnCpkqaYl6nHroaMg,4.698595,Tumerico,705.0
88194,_aKr7POnacW_VizRKBpCiA,4.681153,Blues City Deli,991.0
56736,NDwoKO79_T49UEKVDlHd3A,4.680989,Sustainable Wine Tours,358.0
22416,8QqnRpM-QxGsjDNuu0E57A,4.677058,Carlillos Cocina,799.0
104116,gP_oWJykA2RocIs_GurKWQ,4.671194,Yats,623.0
59589,OR7VJQ3Nk1wCcIbPN4TCQQ,4.648217,Smiling With Hope Pizza,526.0
121351,nk96iwJV1_p2HECYQW1ysA,4.640838,French Quarter Phantoms,1481.0
42231,GuzbBFraIq-fbkjfvaTRvg,4.623709,Mesa Verde,1796.0
48406,JbzvJJolDBT1614qo2Yiaw,4.620155,Nelson's Green Brier Distillery,545.0


### Ciudad

In [107]:
def city_match_score(
    city: str,
    preferred_cities = None
) -> float:
    
    if preferred_cities is None:
        return 1.0
    
    if isinstance(preferred_cities, str):
        preferred_cities = [preferred_cities]
    
    preferred_cities = [c.lower().strip() for c in preferred_cities]
    
    if str(city).lower().strip() in preferred_cities:
        return 1.1  # boost de 10% si la ciudad coincide
    
    return 0.0

### Intencion

In [108]:
# "Quiero comer"
FOOD_INTENT = {
    "restaurants",
    "food",
    "pizza",
    "burgers",
    "mexican",
    "italian",
    "sushi_bars",
    "japanese",
    "chinese",
    "thai",
    "korean",
    "vietnamese",
    "seafood",
    "steakhouses",
    "barbeque",
    "sandwiches",
    "salad",
    "tacos",
    "ramen",
    "poke",
    "diners",
    "delis",
    "food_trucks",
    "fast_food",
    "breakfast_&_brunch",
    "specialty_food",
    "chicken_wings",
    "food_delivery_services",
    "middle_eastern",
    "caribbean",
    'local_flavor',
}

# "Quiero un café o algo ligero"
CAFE_SNACK_INTENT = {
    "coffee_&_tea",
    "cafes",
    "bakeries",
    "bubble_tea",
    "desserts",
    "ice_cream_&_frozen_yogurt",
    "donuts",
    "bagels",
    "juice_bars_&_smoothies",
    "tea_rooms",
    "gelato",
    "macarons",
    "sandwiches"
}

RESTAURANT_CORE =  FOOD_INTENT |  CAFE_SNACK_INTENT

df_bussinesses_food = df_businesses[
    df_businesses['category_list'].apply(lambda cats: any(cat in RESTAURANT_CORE for cat in cats))
].copy()
print(f"Negocios relacionados con comida: {len(df_bussinesses_food)}")

df_bussiness_restaurants = df_businesses[
    df_businesses['category_list'].apply(lambda cats: 'restaurants' in cats)
].copy()
print(f"Negocios relacionados con restaurantes: {len(df_bussiness_restaurants)}")

both_food_and_restaurants = set(df_bussinesses_food['business_id']).intersection(set(df_bussiness_restaurants['business_id']))
print(f"Negocios relacionados con comida y restaurantes: {len(both_food_and_restaurants)}")
just_food_not_restaurants = set(df_bussinesses_food['business_id']) - set(df_bussiness_restaurants['business_id'])
print(f"Negocios relacionados con comida pero no restaurantes: {len(just_food_not_restaurants)}")
just_restaurants_not_food = set(df_bussiness_restaurants['business_id']) - set(df_bussinesses_food['business_id'])
print(f"Negocios relacionados con restaurantes pero no comida: {len(just_restaurants_not_food)}")


# marcamos como restaurant aquellos negocios que tengan la categoría "restaurants" 
# o que tengan alguna categoría relacionada con comida pero no sean restaurantes
df_businesses['is_restaurant'] = df_businesses['business_id'].apply(
    lambda bid: 1 if (bid in both_food_and_restaurants or bid in just_food_not_restaurants) else 0
)

Negocios relacionados con comida: 65478
Negocios relacionados con restaurantes: 52268
Negocios relacionados con comida y restaurantes: 52268
Negocios relacionados con comida pero no restaurantes: 13210
Negocios relacionados con restaurantes pero no comida: 0


In [109]:
# Quiero algo familiar
FAMILY_INTENT = {
    "kids_activities",
    "playgrounds",
    "children's_museums",
    "parks",
    "trampoline_parks",
    "museums",
    "zoos",
    "petting_zoos",
    "amusement_parks",
    "mini_golf",
    "bowling",
    "ice_cream_&_frozen_yogurt",
    "pizza",
    "books",
    "pets"
}


# Quiero hacer ejercicio o alguna actividad
ACTIVE_LIFE_INTENT = {
    "active_life",
    "gyms",
    "yoga",
    "pilates",
    "parks",
    "hiking",
    "bike_rentals",
    "bikes",
    "cycling_classes",
    "martial_arts",
    "boxing",
    "climbing",
    "rock_climbing",
    "swimming_pools",
    "golf",
    "tennis",
    "pickleball",
    "shopping",
    "beauty_&_spas",
    "arts_&_entertainment",
    "massage",
    "books",
    "thrift_stores",
}

NIGHTLIFE_INTENT = {
    "nightlife",
    "bars",
    "beer",
    "lounges",
    "karaoke",
    "comedy_clubs",
    "sports_bars",
    "pubs",
    "speakeasies",
    "wine_bars",
    "wine_&_spirits",
    "cocktail_bars",
    "music_venues"
}

SERVICES_INTENT = {
    'home_services', 'automotive', 'health_&_medical', 'local_services',
       'auto_repair', 'hotels_&_travel', 'event_planning_&_services',
       'real_estate', 'doctors', 'hotels', 'dentists', 'tires',
       'oil_change_stations', 'professional_services', 'general_dentistry',
       'apartments', 'auto_parts_&_supplies', 'contractors', 'car_dealers',
       'cosmetic_dentists', 'financial_services', 'public_services_&_government',
       'banks_&_credit_unions', 'education', 'insurance',
       'religious_organizations', 'libraries', 'churches', 'auto_insurance',
       'specialty_schools', 'life_insurance',
       'home_&_rental_insurance', 'post_offices', 'colleges_&_universities',
       'landmarks_&_historical_buildings', 'mass_media',
       'departments_of_motor_vehicles', 'elementary_schools', "beauty_&_spas"}

preferred_categories = FOOD_INTENT | CAFE_SNACK_INTENT | FAMILY_INTENT | ACTIVE_LIFE_INTENT | NIGHTLIFE_INTENT | SERVICES_INTENT
print(f"Total preferred categories: {len(preferred_categories)}")
bussines_coverage = df_businesses["category_list"].apply(lambda cats: any(cat in preferred_categories for cat in cats)).mean() * 100
print(f"Percentage of businesses that match at least one preferred category: {bussines_coverage:.2f}%")

does_not_exist = (preferred_categories) - set(unique_categories)
print(f"Preferred categories that do not exist in the dataset: {len(does_not_exist)}")
print(does_not_exist)

Total preferred categories: 128
Percentage of businesses that match at least one preferred category: 100.00%
Preferred categories that do not exist in the dataset: 0
set()


In [110]:
most_common_categories = df_businesses["category_list"].explode().value_counts().head(200)
print("Most common categories after cleaning:")
display(most_common_categories)

Most common categories after cleaning:


category_list
restaurants       52268
food              27781
shopping          24395
home_services     14356
beauty_&_spas     14292
                  ...  
cheesesteaks        614
thrift_stores       610
tattoo              609
health_markets      601
caribbean           590
Name: count, Length: 200, dtype: int64

### Revisar si está abierto a cierta hora

In [111]:
from typing import Optional

def _is_hour_inside_range(time_range: str, hour: int) -> bool:
    """
    Evalúa si una hora entera está dentro de un rango tipo '10:00-21:00'.
    También maneja rangos que cruzan medianoche, como '18:00-02:00'.
    """

    if not isinstance(time_range, str) or "-" not in time_range:
        return True

    start_time, end_time = time_range.split("-")

    start_hour = int(start_time.split(":")[0])
    end_hour = int(end_time.split(":")[0])

    # Abierto 24 horas, ejemplo: 0:00-0:00
    if start_hour == end_hour:
        return True

    # Caso normal: 10:00-21:00
    if start_hour < end_hour:
        return start_hour <= hour < end_hour

    # Caso cruza medianoche: 18:00-02:00
    return hour >= start_hour or hour < end_hour


def is_open_at_context(
    hours_dict,
    day_list: Optional[list[str]] = None,
    hour: Optional[int] = None,
    filter_by_day: bool = False,
    filter_by_hour: bool = False
) -> bool:
    """
    Permite filtrar por día, por hora, por ambos o por ninguno.
    """

    # Si no quiero filtrar por día ni por hora, no excluyo el negocio
    if not filter_by_day and not filter_by_hour:
        return True

    # Si no hay diccionario de horarios, no excluyo el negocio
    if not isinstance(hours_dict, dict):
        return True

    # Caso 1: filtrar por día, pero no por hora
    if filter_by_day and not filter_by_hour:
        if day_list is None:
            return True
        # Si el día es una lista, revisamos si alguno de los días está en el diccionario de horarios
        return any(d in hours_dict for d in day_list)

    # Caso 2: filtrar por hora, pero no por día
    # Aquí revisamos si está abierto a esa hora en al menos un día
    if filter_by_hour and not filter_by_day:
        if hour is None:
            return True

        return any(
            _is_hour_inside_range(time_range, hour)
            for time_range in hours_dict.values()
        )

    # Caso 3: filtrar por día y por hora
    if filter_by_day and filter_by_hour:
        if day_list is None or hour is None:
            return True
        open_at_day = any(d in hours_dict for d in day_list)
        open_at_day_and_hour = False
        if open_at_day:
            open_at_day_and_hour = any(_is_hour_inside_range(hours_dict[day], hour) for day in day_list)
        return open_at_day_and_hour

    return True

### Time Context

In [112]:
TIME_CONTEXT = {
    "morning": {
        "hours": range(6, 11),
        "boost_categories": {
            "breakfast_&_brunch": 1.5,
            "coffee_&_tea": 1.5,
            "cafes": 1.4,
            "bakeries": 1.4,
            "bagels": 1.3,
            "donuts": 1.3,
            "juice_bars_&_smoothies": 1.2
        }
    },
    "lunch": {
        "hours": range(11, 15),
        "boost_categories": {
            "restaurants": 1.2,
            "sandwiches": 1.4,
            "salad": 1.3,
            "soup": 1.2,
            "fast_food": 1.2,
            "food_trucks": 1.3,
            "tacos": 1.3,
            "poke": 1.3,
            "sushi_bars": 1.2
        }
    },
    "afternoon": {
        "hours": range(15, 18),
        "boost_categories": {
            "coffee_&_tea": 1.4,
            "cafes": 1.3,
            "desserts": 1.4,
            "ice_cream_&_frozen_yogurt": 1.4,
            "bakeries": 1.2,
            "bubble_tea": 1.3,
            "juice_bars_&_smoothies": 1.2
        }
    },
    "dinner": {
        "hours": range(18, 23),
        "boost_categories": {
            "restaurants": 1.2,
            "italian": 1.3,
            "sushi_bars": 1.3,
            "japanese": 1.2,
            "mexican": 1.2,
            "thai": 1.2,
            "steakhouses": 1.4,
            "seafood": 1.3,
            "mediterranean": 1.2,
            "pizza": 1.2,
            "barbeque": 1.3,
            "french": 1.3
        }
    },
    "late_night": {
        "hours": list(range(23, 24)) + list(range(0, 6)),
        "boost_categories": {
            "pizza": 1.5,
            "fast_food": 1.4,
            "bars": 1.4,
            "pubs": 1.3,
            "sports_bars": 1.3,
            "lounges": 1.2,
            "food_trucks": 1.2,
            "diners": 1.3
        }
    }
}

def get_time_bucket(hour):
    if 6 <= hour < 11:
        return "morning"
    elif 11 <= hour < 15:
        return "lunch"
    elif 15 <= hour < 18:
        return "afternoon"
    elif 18 <= hour < 23:
        return "dinner"
    else:
        return "late_night"
    

def context_score_by_hour(cat_list, hour):
    bucket = get_time_bucket(hour)
    boost_dict = TIME_CONTEXT[bucket]["boost_categories"]

    boost = 1.0

    for cat in cat_list:
        if cat in boost_dict:
            #print(f"Category '{cat}' gets a boost of {boost_dict[cat]} in the '{bucket}' time bucket.")
            boost = max(boost, boost_dict.get(cat, 1.0))
    #print(f"Final boost for categories {cat_list} at hour {hour}: {boost:.2f}")
    # number between 1.0 and 1.5, where 1.0 means no boost and 1.5 means max boost
    return boost


# Ejemplo de uso
boost = context_score_by_hour(
    ["coffee_&_tea", "bakeries"],
    hour=8
)
print(f"Contextual boost for morning coffee: {boost:.2f}")

Contextual boost for morning coffee: 1.50


### Categories

In [113]:
def preferred_category_score(
    cat_list: list[str],
    preferred_categories: list[str],
) -> float:
    
    
    if preferred_categories is None:
        return 1.0
    
    cat_list_clean = {
        str(cat).lower().strip()
        for cat in cat_list
    }
    
    matches = cat_list_clean.intersection(preferred_categories)
    
    if len(matches) == 0:
        return 0.0
    
    return len(matches) / len(preferred_categories)

# Business Context Recomendation

La estrategia planteada es:
1. Filtrar negocios no vistos por el usuario y que esten abiertos.
2. Si aplica, filtrar ciudades preferidas.
3. Si aplica, filtrar categorias preferidas.
4. Si aplica, filtrar dia/hora preferido.
5. Calcular predicción del modelo SVD++.
6. Calcular score de popularidad.
7. Calcure score por ciudad.
8. Calcular time-context score.
9. Calcular categories-score.
10. Ponderar score final.
11. Reordenar ranking de recomendación.

In [114]:
def normalize_rating(pred_rating):
    # el pred_rating de SVD++ está entre 1 y 5, lo normalizamos a [0, 1] para que sea compatible con los otros scores
    return (pred_rating - 1) / 4

def normalize_category_list(
    categories: Optional[Union[str, list[str]]]
) -> Optional[list[str]]:
    
    if categories is None:
        return None
    
    if isinstance(categories, str):
        categories = [categories]
    
    return [
        category.lower().strip()
        for category in categories
    ]

# Función de ponderación final para combinar los diferentes scores en un único score final
def restaurant_ponderate_score(
    model_score: float,
    popularity_score: float,
    hour_score: float = 0.0,
    type_score: float = 0.0,
    city_score: float = 0.0,
    category_score: float = 0.0,
    use_hour: bool = True,
    use_city: bool = True,
    use_category: bool = True
) -> float:
    
    weights = {
        "model_score": 0.60,
        "popularity_score": 0.10,
        #"type_score": 0.10,
        "hour_score": 0.10 if use_hour else 0.0,
        "city_score": 0.05 if use_city else 0.0,
        "category_score": 0.05 if use_category else 0.0,
    }

    total_weight = sum(weights.values())

    normalized_weights = {
        key: value / total_weight
        for key, value in weights.items()
    }

    score = (
        normalized_weights["model_score"] * model_score +
        normalized_weights["popularity_score"] * popularity_score +
        #normalized_weights["type_score"] * type_score +
        normalized_weights["hour_score"] * hour_score +
        normalized_weights["city_score"] * city_score +
        normalized_weights["category_score"] * category_score
    )

    return score * 5  # Escalamos de vuelta a [0, 5]

# MAIN FUNCTION
def recommend_contextual_businesses(
    user_id: str,
    model,
    businesses_df: pd.DataFrame,
    reviews_df: pd.DataFrame,
    top_n: int = 10,
    preferred_cities: Optional[Union[str, list[str]]] = None,
    filter_by_city: bool = False,
    preferred_categories: Optional[Union[str, list[str]]] = None,
    filter_by_categories: bool = False,
    preferred_hour: Optional[int] = 20, # entero entre 0 y 23
    filter_by_open_hour: bool = False,
    preferred_day: Optional[str] = None,
    filter_by_open_day: bool = False,
    
) -> pd.DataFrame:
    
    preferred_categories = normalize_category_list(preferred_categories)
    preferred_cities = normalize_category_list(preferred_cities)
    
    # 1. Negocios ya vistos por el usuario
    seen_businesses = set(
        reviews_df.loc[
            reviews_df["user_id"] == user_id,
            "business_id"
        ]
    )

    # 2. Filtro base: bussiness abiertos oficialmente y no vistos por el usuario
    base_mask = (
        ~businesses_df["business_id"].isin(seen_businesses) &
        (businesses_df["is_open"] == 1)
    )

    candidates = businesses_df[base_mask].copy()

    # 3. Filtro opcional por ciudad
    # Si activo el filtro, elimina de la lista de candidatos aquellos negocios cuya ciudad no esté en preferred_cities
    if preferred_cities is not None and filter_by_city:
        
        candidates = candidates[
            candidates["city"]
            .str.lower()
            .str.strip()
            .isin(preferred_cities)
        ].copy()

    # 4. Filtro opcional por categorías deseadas
    # Si activo el filtro, elimina de la lista de candidatos aquellos negocios que no tengan al menos una categoría en preferred_categories
    if preferred_categories is not None and filter_by_categories:
        candidates = candidates[
            candidates["category_list"].apply(
                lambda cat_list: len(
                    {
                        str(cat).lower().strip()
                        for cat in cat_list
                    }.intersection(preferred_categories)
                ) > 0
            )
        ].copy()

    # 5. Filtro opcional por día, hora o ambos
    # Si activo el filtro, elimina de la lista de candidatos aquellos negocios que no estén abiertos en el listado de dias y/o hora preferidos
    if "hours" in candidates.columns:
        candidates = candidates[
            candidates["hours"].apply(
                lambda hours_dict: is_open_at_context(
                    hours_dict=hours_dict,
                    day_list=preferred_day,
                    hour=preferred_hour,
                    filter_by_day=filter_by_open_day,
                    filter_by_hour=filter_by_open_hour
                )
            )
        ].copy()

    if candidates.empty:
        return pd.DataFrame()

    # 6. Score del modelo SVD++
    candidates["pred_rating"] = candidates["business_id"].apply(
        lambda business_id: model.predict(user_id, business_id).est
    )
    # normalizamos el pred_rating a [0, 1] para que sea compatible con los otros scores
    candidates["model_score"] = candidates["pred_rating"].apply(normalize_rating)

    # 7. Score de popularidad
    candidates["popularity_score"] = candidates["pop_score"].fillna(0)

    # 9. Score por hora
    if preferred_hour is not None:
        candidates["hour_score"] = candidates["category_list"].apply(
            lambda cat_list: context_score_by_hour(cat_list, preferred_hour)
        )
    else:
        candidates["hour_score"] = 1.0

    # 10. Score por ciudad
    candidates["city_score"] = candidates["city"].apply(
        lambda city: city_match_score(
            city=city,
            preferred_cities=preferred_cities
        )
    )

    # 11. Score por categorías deseadas
    candidates["category_score"] = candidates["category_list"].apply(
        lambda cat_list: preferred_category_score(
            cat_list=cat_list,
            preferred_categories=preferred_categories
        )
    )

    # 12. Score final
    candidates["final_score"] = candidates.apply(
        lambda row: restaurant_ponderate_score(
            model_score=row["model_score"],
            popularity_score=row["popularity_score"],
            hour_score=row["hour_score"],
            city_score=row["city_score"],
            category_score=row["category_score"],
            use_hour=preferred_hour is not None,
            use_city=preferred_cities is not None,
            use_category=preferred_categories is not None
        ),
        axis=1
    )

    recommendations = candidates.sort_values(
        "final_score",
        ascending=False
    ).head(top_n)

    columns_to_return = [
        "business_id",
        "name",
        "city",
        "category_list",
        "pred_rating",
        "model_score",
        "popularity_score",
        "hour_score",
        #"type_score",
        #"restaurant_type",
        "city_score",
        "category_score",
        "final_score"
    ]

    if "hours" in recommendations.columns:
        columns_to_return.append("hours")

    return recommendations[columns_to_return]

no filtrar ni por dia ni por hora ni por ciudad

In [115]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model=model,
    businesses_df=df_businesses,
    reviews_df=df_reviews,
    filter_by_open_day=False,
    filter_by_open_hour=False,
    filter_by_city=False,
    filter_by_categories=False,
)
recommendations

,business_id,name,city,category_list,pred_rating,model_score,popularity_score,hour_score,city_score,category_score,final_score,hours
125597,OR7VJQ3Nk1wCcIbPN4TCQQ,Smiling With Hope Pizza,reno,"[italian, restaurants, salad, pizza]",4.763632,0.940908,0.626807,1.3,1.0,1.0,4.732660,"{'Monday': '0:0-0:0', 'Wednesday': '17:0-20:0'..."
21460,gjHB6p19V_bS-R5lUZux4A,South Pacific Grill,brandon,"[asian_fusion, food_trucks, food, restaurants,...",4.924525,0.981131,0.480947,1.2,1.0,1.0,4.729833,"{'Wednesday': '11:0-14:0', 'Thursday': '11:0-1..."
143157,ytynqOUb3hjKeJfRj5Tshw,Reading Terminal Market,philadelphia,"[candy_stores, shopping, department_stores, fa...",4.503550,0.875888,0.960821,1.3,1.0,1.0,4.697592,"{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."
69859,TE2IEDNV0RcI6s1wTOP4fg,Tortilleria San Roman,philadelphia,"[convenience_stores, italian, specialty_food, ...",4.807283,0.951821,0.504459,1.3,1.0,1.0,4.697115,"{'Monday': '0:0-0:0', 'Tuesday': '9:0-17:0', '..."
7801,2KIDQyTh-HzLxOUEDqtDBg,Mazzaro's Italian Market,saint petersburg,"[specialty_food, delis, coffee_roasteries, but...",4.624836,0.906209,0.778081,1.3,1.0,1.0,4.697085,"{'Monday': '9:0-17:0', 'Tuesday': '9:0-17:0', ..."
12307,_aKr7POnacW_VizRKBpCiA,Blues City Deli,saint louis,"[delis, bars, restaurants, nightlife, pubs, am...",4.731060,0.932765,0.715395,1.2,1.0,1.0,4.694991,"{'Monday': '0:0-0:0', 'Tuesday': '10:30-15:0',..."
31165,ctHjyadbDQAtUFfkcAFEHw,Zahav,philadelphia,"[nightlife, bars, food, ethnic_food, middle_ea...",4.623072,0.905768,0.873434,1.2,1.0,1.0,4.692526,"{'Monday': '0:0-0:0', 'Tuesday': '16:45-21:30'..."
145497,9xdXS7jtWjCVzL4_oPGv9A,GW Fins,new orleans,"[seafood, gluten-free, vegetarian, restaurants]",4.576817,0.894204,0.841678,1.3,1.0,1.0,4.691815,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-21:30',..."
94147,SFN5g5eexukcaZkzoiYQYg,Kounter Kulture,st. louis,"[korean, asian_fusion, salad, japanese, soup, ...",4.829883,0.957471,0.561247,1.2,1.0,1.0,4.691295,{'Monday': '0:0-0:0'}
97443,42dVj5q-LMx_iJxcq5Fzng,Vida,indianapolis,"[restaurants, local_flavor, american_(new), sa...",4.681792,0.920448,0.570286,1.4,1.0,1.0,4.683109,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-22:0', ..."


filtrar por ciudad

In [116]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model=model,
    businesses_df=df_businesses,
    reviews_df=df_reviews,
    preferred_cities=["Philadelphia", "Tampa", "Nashville"],
    filter_by_city=True,
    filter_by_open_day=False,
    filter_by_open_hour=False,
    top_n=10
)

recommendations[["name", "city", "category_list", "pred_rating", "city_score", "final_score"]]

,name,city,category_list,pred_rating,city_score,final_score
143157,Reading Terminal Market,philadelphia,"[candy_stores, shopping, department_stores, fa...",4.503550,1.1,4.744792
69859,Tortilleria San Roman,philadelphia,"[convenience_stores, italian, specialty_food, ...",4.807283,1.1,4.744344
31165,Zahav,philadelphia,"[nightlife, bars, food, ethnic_food, middle_ea...",4.623072,1.1,4.740025
27407,Castellino's,philadelphia,"[restaurants, food, sardinian, italian, delis,...",4.886669,1.1,4.723505
28139,KC Carpet and Upholstery Cleaners,philadelphia,"[grout_services, home_cleaning, carpet_cleanin...",4.956145,1.1,4.710736
58300,Yolk White & Associates,tampa,"[food_stands, restaurants, food, coffee_&_tea,...",4.829912,1.1,4.709149
118708,The Mediterranean Chickpea,tampa,"[restaurants, vegetarian, mediterranean, vegan]",4.820704,1.1,4.684850
142749,Terra Gaucha Brazilian Steakhouse - Tampa,tampa,"[steakhouses, restaurants, buffets, brazilian,...",4.612416,1.1,4.681705
95158,Big Al's Deli,nashville,"[restaurants, delis, southern]",4.738893,1.1,4.672554
149507,Restaurant Ambra,philadelphia,"[italian, restaurants]",4.827658,1.1,4.657981


Ciudad como boost, no como filtro
- Aquí pueden salir restaurantes de otras ciudades, pero las ciudades preferidas reciben mejor `city_score`.

In [117]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model=model,
    businesses_df=df_businesses,
    reviews_df=df_reviews,
    preferred_cities=["tampa"],
    filter_by_city=False,  # no filtramos por ciudad, solo le damos un boost a las que están en esas ciudades
    filter_by_open_day=False,
    filter_by_open_hour=False,
    top_n=10
)

recommendations[["name", "city", "category_list", "pred_rating", "city_score", "final_score"]]

,name,city,category_list,pred_rating,city_score,final_score
58300,Yolk White & Associates,tampa,"[food_stands, restaurants, food, coffee_&_tea,...",4.829912,1.1,4.709149
118708,The Mediterranean Chickpea,tampa,"[restaurants, vegetarian, mediterranean, vegan]",4.820704,1.1,4.684850
142749,Terra Gaucha Brazilian Steakhouse - Tampa,tampa,"[steakhouses, restaurants, buffets, brazilian,...",4.612416,1.1,4.681705
125264,Sulphur Springs Sandwiches Shop,tampa,"[sandwiches, vegetarian, delis, cafes, restaur...",4.720233,1.1,4.643845
2481,Yah Mon,tampa,"[restaurants, caribbean]",4.629834,1.1,4.624069
113362,Oak and Ola,tampa,"[nightlife, modern_european, cocktail_bars, ba...",4.664153,1.1,4.623502
117431,Magdalena's Pizzeria,tampa,"[pizza, italian, restaurants, filipino, chicke...",4.673499,1.1,4.609308
136068,Spaddy's Coffee,tampa,"[breakfast_&_brunch, coffee_&_tea, food, food_...",4.767720,1.1,4.608527
149077,Pure Kitchen Organic Vegan,tampa,"[vegetarian, juice_bars_&_smoothies, organic_s...",4.767618,1.1,4.607810
36987,Lolis Mexican Cravings,tampa,"[ethnic_food, restaurants, specialty_food, mex...",4.532039,1.1,4.598011


filtrar solo por dia

In [118]:
preferred_day = "Friday"
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model=model,
    businesses_df=df_businesses,
    reviews_df=df_reviews,
    preferred_day=[preferred_day],
    filter_by_open_day=True,
    filter_by_open_hour=False
)

recommendations[["name", "pred_rating", "final_score", "hours"]]

,name,pred_rating,final_score,hours
125597,Smiling With Hope Pizza,4.763632,4.732660,"{'Monday': '0:0-0:0', 'Wednesday': '17:0-20:0'..."
21460,South Pacific Grill,4.924525,4.729833,"{'Wednesday': '11:0-14:0', 'Thursday': '11:0-1..."
143157,Reading Terminal Market,4.503550,4.697592,"{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."
69859,Tortilleria San Roman,4.807283,4.697115,"{'Monday': '0:0-0:0', 'Tuesday': '9:0-17:0', '..."
7801,Mazzaro's Italian Market,4.624836,4.697085,"{'Monday': '9:0-17:0', 'Tuesday': '9:0-17:0', ..."
12307,Blues City Deli,4.731060,4.694991,"{'Monday': '0:0-0:0', 'Tuesday': '10:30-15:0',..."
31165,Zahav,4.623072,4.692526,"{'Monday': '0:0-0:0', 'Tuesday': '16:45-21:30'..."
145497,GW Fins,4.576817,4.691815,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-21:30',..."
97443,Vida,4.681792,4.683109,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-22:0', ..."
27407,Castellino's,4.886669,4.674974,"{'Monday': '0:0-0:0', 'Tuesday': '12:0-17:0', ..."


filtrar solo por hora

In [119]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model=model,
    businesses_df=df_businesses,
    reviews_df=df_reviews,
    preferred_hour=20, # int entre 0 y 23
    filter_by_open_day=False,
    filter_by_open_hour=True
)
recommendations[["name", "pred_rating", "final_score", "hours"]]

,name,pred_rating,final_score,hours
125597,Smiling With Hope Pizza,4.763632,4.732660,"{'Monday': '0:0-0:0', 'Wednesday': '17:0-20:0'..."
69859,Tortilleria San Roman,4.807283,4.697115,"{'Monday': '0:0-0:0', 'Tuesday': '9:0-17:0', '..."
12307,Blues City Deli,4.731060,4.694991,"{'Monday': '0:0-0:0', 'Tuesday': '10:30-15:0',..."
31165,Zahav,4.623072,4.692526,"{'Monday': '0:0-0:0', 'Tuesday': '16:45-21:30'..."
145497,GW Fins,4.576817,4.691815,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-21:30',..."
94147,Kounter Kulture,4.829883,4.691295,{'Monday': '0:0-0:0'}
97443,Vida,4.681792,4.683109,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-22:0', ..."
27407,Castellino's,4.886669,4.674974,"{'Monday': '0:0-0:0', 'Tuesday': '12:0-17:0', ..."
28139,KC Carpet and Upholstery Cleaners,4.956145,4.661407,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-17:0', '..."
29873,Barista Del Barrio,4.785679,4.656982,{'Monday': '0:0-0:0'}


filtrar por dia y hora

In [120]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model=model,
    businesses_df=df_businesses,
    reviews_df=df_reviews,
    preferred_day=["Friday"],
    preferred_hour=20,
    filter_by_open_day=True,
    filter_by_open_hour=True
)
recommendations[["name", "pred_rating", "final_score", "hours"]]

,name,pred_rating,final_score,hours
31165,Zahav,4.623072,4.692526,"{'Monday': '0:0-0:0', 'Tuesday': '16:45-21:30'..."
145497,GW Fins,4.576817,4.691815,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-21:30',..."
142749,Terra Gaucha Brazilian Steakhouse - Tampa,4.612416,4.630562,"{'Monday': '17:0-21:30', 'Tuesday': '17:0-21:3..."
149507,Restaurant Ambra,4.827658,4.605354,"{'Thursday': '18:0-21:0', 'Friday': '18:0-21:0..."
6077,Sushi Ushi,4.685311,4.596373,"{'Tuesday': '16:30-21:30', 'Wednesday': '16:30..."
149067,St Yared Ethiopian Restaurant,4.753437,4.595674,"{'Tuesday': '11:0-20:0', 'Wednesday': '11:0-20..."
76680,Corazon Cocina,4.574828,4.593713,"{'Monday': '0:0-0:0', 'Tuesday': '11:0-21:0', ..."
15939,The Black Pearl,4.658250,4.592623,"{'Monday': '17:0-21:0', 'Tuesday': '17:0-22:0'..."
76386,Kuma Sushi & Asian Fusion,4.760502,4.592583,"{'Tuesday': '11:30-21:0', 'Wednesday': '11:30-..."
56003,Ken Love's BYOB,4.641230,4.582903,"{'Wednesday': '17:0-22:0', 'Thursday': '17:0-2..."


filtrar por categorias

In [121]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model=model,
    businesses_df=df_businesses,
    reviews_df=df_reviews,
    preferred_day=["Friday"],
    preferred_hour=20,
    preferred_categories=["pizza", "italian", "mexican"],
    filter_by_open_day=True,
    filter_by_open_hour=True
)
recommendations[["name", "pred_rating", "final_score", "hours", "category_list", "category_score"]]

,name,pred_rating,final_score,hours,category_list,category_score
103613,Five Points Pizza,4.494342,4.486743,"{'Monday': '0:0-0:0', 'Tuesday': '11:0-22:0', ...","[nightlife, bars, italian, restaurants, pizza,...",0.666667
117431,Magdalena's Pizzeria,4.673499,4.481857,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-21:0', ...","[pizza, italian, restaurants, filipino, chicke...",0.666667
149507,Restaurant Ambra,4.827658,4.432490,"{'Thursday': '18:0-21:0', 'Friday': '18:0-21:0...","[italian, restaurants]",0.333333
68767,Louie,4.591461,4.430481,"{'Monday': '0:0-0:0', 'Tuesday': '11:0-14:0', ...","[italian, pizza, restaurants]",0.666667
76680,Corazon Cocina,4.574828,4.421533,"{'Monday': '0:0-0:0', 'Tuesday': '11:0-21:0', ...","[restaurants, breakfast_&_brunch, latin_americ...",0.333333
31165,Zahav,4.623072,4.416495,"{'Monday': '0:0-0:0', 'Tuesday': '16:45-21:30'...","[nightlife, bars, food, ethnic_food, middle_ea...",0.000000
145497,GW Fins,4.576817,4.415825,"{'Monday': '0:0-0:0', 'Tuesday': '17:0-21:30',...","[seafood, gluten-free, vegetarian, restaurants]",0.000000
31684,Noble Crust,4.387064,4.389893,"{'Monday': '16:0-22:0', 'Tuesday': '16:0-22:0'...","[breakfast_&_brunch, pizza, southern, italian,...",0.666667
149149,Wm Mulherin's Sons,4.450509,4.386246,"{'Tuesday': '17:0-22:0', 'Wednesday': '17:0-22...","[restaurants, tapas/small_plates, italian, sze...",0.666667
135274,Collegeville Italian Bakery Pizzeria Napoletana,4.508124,4.385329,"{'Monday': '8:0-20:0', 'Tuesday': '8:0-20:0', ...","[italian, restaurants, pizza, bakeries, food]",0.666667
